# Fine-tune PhoASR-whisper-small cho robot dẫn đường bệnh viện (Colab GPU)

Notebook này fine-tune **PhoASR-whisper-small** trên bộ giọng nói của dự án, kèm **augmentation ồn bệnh viện mạnh** và các augmentation khác ở mức nhẹ. Logic giống `scripts/train_asr_finetune.py`, được chỉnh cho GPU: batch lớn hơn, mixed precision, và lưu thẳng lên Google Drive.

**Chuẩn bị (làm 1 lần):**
1. Chọn `Runtime → Change runtime type → GPU` (T4 là đủ; A100/L4 nhanh hơn).
2. Đưa zip dữ liệu lên Colab: `data/processed/voice_dataset_v2_colab.zip` (bộ mới: train bằng giọng clone + Bud500, test bằng toàn bộ data thật) hoặc `voice_dataset_colab.zip` (bộ cũ, 151 MB). Chọn bộ ở biến `DATASET` bước 2 theo một trong các cách ở bước 2 (dễ nhất: kéo thả vào `MyDrive/hospital_asr/` trên Google Drive). Chỉ cần làm lần đầu.
3. `Runtime → Run all`.

**Mọi thứ của một lần chạy** nằm trong `MyDrive/hospital_asr/runs/<YYYYmmdd_HHMMSS>_ASR-<run_name>/`:

| Đường dẫn | Nội dung |
|---|---|
| `config.json`, `train.log` | tham số, log đầy đủ |
| `metrics_steps.csv`, `metrics_epochs.csv` | loss/lr/grad-norm theo step; loss + WER/CER val theo epoch |
| `plots/` | loss curve, val WER, learning rate, so sánh base và fine-tuned theo người nói và mức ồn |
| `checkpoints/best/`, `checkpoints/last/` | model + processor (`save_pretrained`) |
| `eval/` | dự đoán từng câu trên test (sạch, ồn 10 dB, ồn 5 dB) và `test_newtext`, `summary.json` |
| `aug_samples/` | vài file đã augment để nghe kiểm tra |

**Mức cần vượt** (model gốc, theo `docs/VOICE_DATASET_EDA.md` §12): WER test 19,1%, WER `_r` 13,3%, WER khi SNR < 30 dB là 47,4%.

> ⚠️ Dữ liệu giọng nói chỉ dùng cho đồ án theo phiếu đồng ý của người thu. Giữ file zip và checkpoint trong Drive cá nhân, không chia sẻ công khai.

## 1. Môi trường

In [12]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q jiwer

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [13]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # phải đặt trước khi import torch
import csv, json, math, random, re, time, logging, subprocess, unicodedata, zipfile, shutil
from datetime import datetime
from pathlib import Path

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, display, Image
from scipy.signal import butter, resample_poly, sosfilt
from torch.utils.data import DataLoader, Dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → GPU"
DEVICE = "cuda"
GPU = torch.cuda.get_device_name(0)
BF16 = torch.cuda.is_bf16_supported()  # A100/L4: bf16; T4: fp16 + GradScaler
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(GPU, f"{VRAM_GB:.1f} GB", "| bf16" if BF16 else "| fp16", "| torch", torch.__version__)

Tesla T4 15.6 GB | bf16 | torch 2.14.1+cu130


## 2. Dữ liệu: tìm zip, upload nếu cần, giải nén

Chọn bộ dữ liệu ở biến `DATASET` (mặc định `voice_dataset_v2`, file `voice_dataset_v2_colab.zip`). Cell dưới tìm zip theo thứ tự sau và dùng nguồn đầu tiên có:

1. **Google Drive**: `MyDrive/hospital_asr/<DATASET>_colab.zip`. Có thể kéo thả file vào Drive trên web trước.
2. **Thanh Files của Colab**: bấm biểu tượng 📁 ở cột trái, kéo thả file zip vào `/content`.
3. **Link Google Drive chia sẻ**: điền `DATA_URL` bên dưới (Drive → chuột phải file → Share → *Anyone with the link* → Copy link).
4. **Nút upload** `files.upload()`: chỉ chạy được trong tab trình duyệt colab.research.google.com. Không chạy được trong VS Code hay khi dùng Run all ở tab nền.

Khi tìm thấy, zip được kiểm tra rồi chép vào Drive (nếu đã kết nối Drive) để các phiên sau dùng luôn.

In [14]:
# "voice_dataset_v2": train/val = giọng clone (OmniVoice) + Bud500, test = TOÀN BỘ data thật (29 người)
# "voice_dataset"   : bản cũ, train/val/test đều là data thật
DATASET = "voice_dataset_v2"
DATA_URL = ""   # tuỳ chọn: link Google Drive chia sẻ của file zip

ZIP_NAME = f"{DATASET}_colab.zip"
DATA = Path("/content") / DATASET              # giải nén ra đĩa local cho nhanh
LOCAL_ZIP = Path("/content") / ZIP_NAME

# Drive: dùng để lưu zip + kết quả. Nếu không mount được (ví dụ chạy từ VS Code) thì lưu ở /content.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/hospital_asr")
except Exception as e:
    print(f"⚠️ Không mount được Google Drive ({e}). Kết quả sẽ lưu ở /content/hospital_asr và MẤT khi phiên kết thúc.")
    DRIVE_DIR = Path("/content/hospital_asr")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DRIVE_DIR / ZIP_NAME
RUNS_DIR = DRIVE_DIR / "runs"

def check_zip(path: Path) -> bool:
    try:
        with zipfile.ZipFile(path) as z:
            names, bad = set(z.namelist()), z.testzip()
    except zipfile.BadZipFile:
        print(f"  {path} không phải zip hợp lệ"); return False
    need = {f"{DATASET}/{n}" for n in ("manifest.csv", "train.jsonl", "val.jsonl", "test.jsonl", "test_newtext.jsonl")}
    if bad or not need <= names:
        print(f"  {path}: file hỏng={bad}, thiếu={sorted(need - names)}"); return False
    return True

def find_zip() -> Path | None:
    # 1) Drive  2) thanh Files của Colab (/content, kể cả tên kiểu "voice_dataset_colab (1).zip")
    for cand in [ZIP_PATH, LOCAL_ZIP, *sorted(Path("/content").glob(f"{DATASET}_colab*.zip"))]:
        if cand.exists() and check_zip(cand):
            return cand
    # 3) link Drive chia sẻ
    if DATA_URL:
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        import gdown
        gdown.download(DATA_URL, str(LOCAL_ZIP), quiet=False, fuzzy=True)
        if LOCAL_ZIP.exists() and check_zip(LOCAL_ZIP):
            return LOCAL_ZIP
    # 4) nút upload (chỉ trong tab trình duyệt Colab)
    try:
        from google.colab import files
        print(f"Chọn file {ZIP_NAME} trên máy:")
        up = files.upload()
        if up:
            name = next(iter(up)); p = Path("/content") / name
            if not p.exists():
                p.write_bytes(up[name])
            if check_zip(p):
                return p
    except Exception as e:
        print(f"  Nút upload không dùng được ở môi trường này ({type(e).__name__}).")
    return None

src = find_zip()
if src is None:
    raise FileNotFoundError(
        f"Không tìm thấy {ZIP_NAME}. Làm MỘT trong các cách:\n"
        f"  a) kéo thả file vào Google Drive tại {ZIP_PATH.parent} rồi chạy lại cell này\n"
        "  b) bấm 📁 ở cột trái Colab, kéo thả file vào /content rồi chạy lại cell này\n"
        "  c) điền DATA_URL = link chia sẻ Google Drive của file zip rồi chạy lại cell này")
print(f"Dùng: {src} ({src.stat().st_size / 1e6:.0f} MB)")
if src != ZIP_PATH and not ZIP_PATH.exists():
    shutil.copy2(src, ZIP_PATH)
    print(f"Đã lưu một bản vào {ZIP_PATH} cho các phiên sau")

if not (DATA / "manifest.csv").exists():
    with zipfile.ZipFile(src) as z:
        z.extractall("/content")
n_audio = sum(1 for p in (DATA / "wav").rglob("*") if p.suffix in (".wav", ".flac"))
print(f"Đã giải nén {n_audio} file audio vào {DATA}")
_m = pd.read_csv(DATA / "manifest.csv")
print(_m.groupby(["split", "source"] if "source" in _m else "split").agg(
    utts=("utt_id", "size"), minutes=("duration_s", lambda s: round(s.sum() / 60, 1)),
    speakers=("speaker_id", "nunique")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dùng: /content/drive/MyDrive/hospital_asr/voice_dataset_v2_colab.zip (646 MB)
Đã giải nén 11689 file audio vào /content/voice_dataset_v2
                    utts  minutes  speakers
split source                               
test  real          1726    117.5        29
train bud500        2444    104.5         1
      synth_clone   3259    232.3       216
      synth_design  3594    210.0        18
val   bud500          56      2.4         1
      synth_clone    301     20.9        24
      synth_design   310     18.2        18


## 3. Cấu hình

Batch được chọn tự động theo VRAM. GPU dưới 20 GB (T4, 14,5 GB): batch 8 × grad-accum 2, bật gradient checkpointing. GPU lớn hơn (L4/A100): batch 16. Effective batch luôn là 16. Nếu vẫn hết bộ nhớ (OOM) thì đặt `batch_size=4, grad_accum=4`, rồi chọn **Runtime → Restart session** và chạy lại từ đầu.

In [15]:
CFG = dict(
    model="Qualcomm-AI-Research/PhoASR-whisper-small",
    run_name="PhoASR-small-hospitalnoise" + ("" if DATASET == "voice_dataset" else "-synth"),
    epochs=10 if DATASET == "voice_dataset" else 6, lr=1e-5, weight_decay=0.01, warmup_ratio=0.1,
    max_grad_norm=1.0, patience=3,             # dừng sớm sau N epoch không cải thiện val WER _r
    freeze_encoder=False,
    modes="r,n", n_weight=0.5,                 # _n có nhãn gần đúng -> loss x0.5 ; "r" để chỉ train _r
    # ồn bệnh viện (mạnh)
    p_noise=0.85, snr_min=0.0, snr_max=20.0,
    # augmentation khác (nhẹ)
    p_speed=0.3, p_gain=0.5, p_lowpass=0.15, spec_augment=0.05,
    test_snrs="10,5", num_workers=2, log_every=10, seed=42,
)
# Chọn theo VRAM (T4 14,5 GB bị OOM ở batch 16)
if VRAM_GB < 20:
    CFG.update(batch_size=8, grad_accum=2, grad_checkpointing=True, eval_batch_size=16)
else:
    CFG.update(batch_size=16, grad_accum=1, grad_checkpointing=False, eval_batch_size=32)
CFG

{'model': 'Qualcomm-AI-Research/PhoASR-whisper-small',
 'run_name': 'PhoASR-small-hospitalnoise-synth',
 'epochs': 6,
 'lr': 1e-05,
 'weight_decay': 0.01,
 'warmup_ratio': 0.1,
 'max_grad_norm': 1.0,
 'patience': 3,
 'freeze_encoder': False,
 'modes': 'r,n',
 'n_weight': 0.5,
 'p_noise': 0.85,
 'snr_min': 0.0,
 'snr_max': 20.0,
 'p_speed': 0.3,
 'p_gain': 0.5,
 'p_lowpass': 0.15,
 'spec_augment': 0.05,
 'test_snrs': '10,5',
 'num_workers': 2,
 'log_every': 10,
 'seed': 42,
 'batch_size': 8,
 'grad_accum': 2,
 'grad_checkpointing': True,
 'eval_batch_size': 16}

## 4. Chuẩn hóa văn bản để tính WER

Giống `scripts/eval_asr_testset.py`: NFC, chữ thường, số đổi thành chữ (410 → bốn trăm mười), `lẻ` = `linh`, bỏ dấu câu.

In [16]:
DIGITS = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]

def num_to_words(n: int) -> str:
    if n < 10:
        return DIGITS[n]
    if n < 100:
        tens, unit = divmod(n, 10)
        head = "mười" if tens == 1 else f"{DIGITS[tens]} mươi"
        tail = {0: "", 1: " mốt" if tens > 1 else " một", 4: " tư" if tens > 1 else " bốn",
                5: " lăm"}.get(unit, f" {DIGITS[unit]}")
        return head + tail
    if n < 1000:
        hund, rest = divmod(n, 100)
        if rest == 0:
            return f"{DIGITS[hund]} trăm"
        return f"{DIGITS[hund]} trăm " + (f"linh {DIGITS[rest]}" if rest < 10 else num_to_words(rest))
    return " ".join(DIGITS[int(c)] for c in str(n))

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = re.sub(r"\d+", lambda m: f" {num_to_words(int(m.group()))} ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\blẻ\b", "linh", text)
    return re.sub(r"\s+", " ", text).strip()

print(normalize("Phòng 410 là phòng gì vậy?"), "|", normalize("phòng ba trăm lẻ bốn"))

phòng bốn trăm mười là phòng gì vậy | phòng ba trăm linh bốn


## 5. Ồn bệnh viện và augmentation

Mỗi mẫu train có xác suất 85% bị trộn 1–2 lớp ồn, SNR 0–20 dB:
- **babble**: 3–6 giọng khác trong tập train, lọc thông thấp, giống tiếng người nói xa ngoài hành lang
- **hvac**: tiếng điều hòa/quạt (nhiễu hồng/nâu lọc thấp) cộng tiếng ù điện 50 Hz
- **beeps**: tiếng bíp máy monitor
- **trolley**: tiếng xe đẩy lạch cạch
- **file**: âm thang máy thu thật (`noise/*.wav`)

Augmentation nhẹ: đổi tốc độ 0,9–1,1×, đổi âm lượng ±6 dB, lọc thông thấp 3,5–7 kHz, SpecAugment.

In [17]:
SR = 16000

def load_wav(path) -> np.ndarray:
    x, sr = sf.read(path, dtype="float32", always_2d=False)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != SR:
        g = math.gcd(sr, SR)
        x = resample_poly(x, SR // g, sr // g).astype(np.float32)
    return x

def rms(x):
    return float(np.sqrt(np.mean(x.astype(np.float64) ** 2) + 1e-12))

def fit_length(x, n, rng):
    if len(x) == 0:
        return np.zeros(n, np.float32)
    if len(x) < n:
        x = np.tile(x, int(np.ceil(n / len(x))) + 1)
    start = rng.integers(0, len(x) - n + 1)
    return x[start:start + n]

def colored_noise(n, rng, beta):
    spec = rng.normal(size=n // 2 + 1) + 1j * rng.normal(size=n // 2 + 1)
    f = np.fft.rfftfreq(n, 1 / SR); f[0] = f[1]
    x = np.fft.irfft(spec / f ** (beta / 2), n)
    return (x / (np.abs(x).max() + 1e-9)).astype(np.float32)

def lowpass(x, cutoff, order=6):
    return sosfilt(butter(order, cutoff, btype="low", fs=SR, output="sos"), x).astype(np.float32)

class HospitalNoise:
    def __init__(self, babble_pool, files):
        self.babble_pool, self.files = babble_pool, files
        self.kinds = ["babble", "hvac", "beeps", "trolley"] + (["file"] if files else [])
        w = {"babble": 3.0, "hvac": 2.0, "beeps": 1.0, "trolley": 1.0, "file": 2.0}
        self.p = np.array([w[k] for k in self.kinds]) / sum(w[k] for k in self.kinds)

    def babble(self, n, rng):
        idx = rng.choice(len(self.babble_pool), size=rng.integers(3, 7), replace=False)
        mix = sum(fit_length(self.babble_pool[i], n, rng) / (rms(self.babble_pool[i]) + 1e-6) for i in idx)
        return lowpass(mix, rng.uniform(3000, 6000), 4)

    def hvac(self, n, rng):
        x = lowpass(colored_noise(n, rng, rng.choice([1.0, 2.0])), rng.uniform(300, 1500), 4)
        t = np.arange(n) / SR
        hum = sum(np.sin(2 * np.pi * h * 50 * t + rng.uniform(0, 6.3)) / h for h in (1, 2, 3))
        return (x / (rms(x) + 1e-9) + 0.3 * hum / (rms(hum) + 1e-9)).astype(np.float32)

    def beeps(self, n, rng):
        t = np.arange(n) / SR
        f0, period, dur = rng.uniform(800, 2000), rng.uniform(0.6, 1.5), rng.uniform(0.08, 0.2)
        gate = ((t + rng.uniform(0, period)) % period) < dur
        return (np.sin(2 * np.pi * f0 * t) * gate + 0.05 * colored_noise(n, rng, 1.0)).astype(np.float32)

    def trolley(self, n, rng):
        x = lowpass(colored_noise(n, rng, 2.0), rng.uniform(200, 800), 4)
        t = np.arange(n) / SR
        return (x * (1 + 0.6 * np.sin(2 * np.pi * rng.uniform(4, 12) * t) ** 2)).astype(np.float32)

    def file(self, n, rng):
        return fit_length(self.files[rng.integers(len(self.files))], n, rng)

    def __call__(self, n, rng, n_layers=None):
        out = np.zeros(n, np.float32)
        for kind in rng.choice(self.kinds, size=n_layers or rng.integers(1, 3), replace=False, p=self.p):
            x = getattr(self, kind)(n, rng)
            out += x / (rms(x) + 1e-9) * rng.uniform(0.5, 1.0)
        return out

def add_noise(x, noise, snr_db):
    y = x + noise * (rms(x) / (rms(noise) * 10 ** (snr_db / 20) + 1e-12))
    peak = np.abs(y).max()
    return (y / peak * 0.89 if peak > 0.89 else y).astype(np.float32)

class Augmenter:
    def __init__(self, noise, cfg):
        self.noise, self.c = noise, cfg

    def __call__(self, x, rng):
        c = self.c
        if rng.random() < c["p_speed"]:
            x = resample_poly(x, 100, int(round(100 * rng.uniform(0.9, 1.1)))).astype(np.float32)
        if rng.random() < c["p_gain"]:
            x = x * 10 ** (rng.uniform(-6, 6) / 20)
        if rng.random() < c["p_lowpass"]:
            x = lowpass(x, rng.uniform(3500, 7000))
        if rng.random() < c["p_noise"]:
            x = add_noise(x, self.noise(len(x), rng), rng.uniform(c["snr_min"], c["snr_max"]))
        return np.clip(x, -1, 1).astype(np.float32)

## 6. Dataset, collator, loss có trọng số

In [18]:
MANIFEST = pd.read_csv(DATA / "manifest.csv").set_index("audio_path")
_dup = MANIFEST.index.duplicated(keep="first")
if _dup.any():   # trùng audio_path -> .loc trả về DataFrame thay vì 1 dòng
    print(f"⚠️ manifest.csv có {_dup.sum()} dòng trùng audio_path, giữ dòng đầu. Ví dụ: {list(MANIFEST.index[_dup][:3])}")
    MANIFEST = MANIFEST[~_dup]

def read_split(name):
    rows = [json.loads(l) for l in open(DATA / f"{name}.jsonl", encoding="utf8")]
    uniq = list({r["audio_filepath"]: r for r in rows}.values())
    if len(uniq) < len(rows):
        print(f"⚠️ {name}.jsonl: bỏ {len(rows) - len(uniq)} dòng trùng audio_filepath")
    rows = uniq
    for r in rows:
        m = MANIFEST.loc[r["audio_filepath"]]
        r.update(utt_id=m.utt_id, snr_db=float(m.snr_db),
                 old_split=str(m.get("old_split", "")), source=str(m.get("source", "real")))
    return rows

class SpeechDataset(Dataset):
    def __init__(self, rows, audio, augment, seed):
        self.rows, self.audio, self.augment, self.seed, self.epoch = rows, audio, augment, seed, 0
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        x = self.audio[r["audio_filepath"]]
        if self.augment:
            x = self.augment(x, np.random.default_rng((self.seed, self.epoch, i)))
        return dict(audio=x, text=r["text"], mode=r["mode"])

class Collator:
    def __init__(self, processor, n_weight):
        self.p, self.n_weight = processor, n_weight
        self.sot = processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
    def __call__(self, batch):
        feats = self.p.feature_extractor([b["audio"] for b in batch], sampling_rate=SR, return_tensors="pt")
        lab = self.p.tokenizer([b["text"] for b in batch], padding=True, return_tensors="pt")
        labels = lab.input_ids.masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.sot).all():      # model tự thêm decoder_start
            labels = labels[:, 1:]
        weights = torch.tensor([self.n_weight if b["mode"] == "n" else 1.0 for b in batch])
        return dict(input_features=feats.input_features, labels=labels, weights=weights)

def weighted_loss(logits, labels, weights):
    tok = torch.nn.functional.cross_entropy(logits.float().transpose(1, 2), labels, ignore_index=-100, reduction="none")
    mask = labels.ne(-100).float()
    per_utt = (tok * mask).sum(1) / mask.sum(1).clamp(min=1)
    return (per_utt * weights).sum() / weights.sum()

⚠️ manifest.csv có 1 dòng trùng audio_path, giữ dòng đầu. Ví dụ: ['wav/synth/syn_t49854364_0.flac']


## 7. Đánh giá và vẽ biểu đồ

In [19]:
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16

@torch.no_grad()
def transcribe(model, processor, waves, bs):
    model.eval()
    out = []
    for i in range(0, len(waves), bs):
        feats = processor.feature_extractor(waves[i:i + bs], sampling_rate=SR, return_tensors="pt")
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            seq = model.generate(input_features=feats.input_features.to(DEVICE), language="vi",
                                 task="transcribe", num_beams=1, use_cache=True)
        out += [s.strip() for s in processor.batch_decode(seq, skip_special_tokens=True)]
    return out

def score(rows, hyps):
    refs, hn = [normalize(r["text"]) for r in rows], [normalize(h) for h in hyps]
    res = {}
    for name, sel in [("all", lambda r: True), ("r", lambda r: r["mode"] == "r"), ("n", lambda r: r["mode"] == "n")]:
        ix = [i for i, r in enumerate(rows) if sel(r)]
        if ix:
            res[f"wer_{name}"] = round(jiwer.wer([refs[i] for i in ix], [hn[i] for i in ix]), 4)
            res[f"cer_{name}"] = round(jiwer.cer([refs[i] for i in ix], [hn[i] for i in ix]), 4)
    res["sent_acc"] = round(float(np.mean([a == b for a, b in zip(refs, hn)])), 4)
    return res

@torch.no_grad()
def val_loss(model, loader):
    model.eval()
    tot, n = 0.0, 0
    for b in loader:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            out = model(input_features=b["input_features"].to(DEVICE), labels=b["labels"].to(DEVICE))
        k = len(b["labels"])
        tot += weighted_loss(out.logits, b["labels"].to(DEVICE), torch.ones(k, device=DEVICE)).item() * k
        n += k
    return tot / max(n, 1)

C1, C2, C3, INK2, GRID, SURF = "#2a78d6", "#eb6834", "#1baf7a", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
                     "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "legend.frameon": False, "axes.titleweight": "bold"})

def plot_curves(run, steps, epochs, show=False):
    pdir = run / "plots"
    if steps:
        s = np.array([r["step"] for r in steps]); loss = np.array([r["train_loss"] for r in steps])
        k = max(1, len(loss) // 30); smooth = np.convolve(loss, np.ones(k) / k, mode="valid")
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(s, loss, color=C1, alpha=0.25, linewidth=1)
        ax.plot(s[k - 1:], smooth, color=C1, linewidth=2, label="train loss (làm mượt)")
        if epochs:
            ax.plot([e["step"] for e in epochs], [e["val_loss"] for e in epochs], color=C2, linewidth=2,
                    marker="o", markersize=6, label="val loss")
        ax.set(title="Loss", xlabel="step", ylabel="cross-entropy"); ax.legend(); fig.tight_layout()
        fig.savefig(pdir / "loss_curve.png", dpi=130); plt.show() if show else None; plt.close(fig)
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(s, [r["lr"] for r in steps], color=C1, linewidth=2)
        ax.set(title="Learning rate", xlabel="step"); fig.tight_layout()
        fig.savefig(pdir / "lr.png", dpi=130); plt.show() if show else None; plt.close(fig)
    if epochs:
        e = [r["epoch"] for r in epochs]
        fig, ax = plt.subplots(figsize=(8, 4))
        for key, c, lab in [("val_wer_all", C1, "val WER"), ("val_wer_r", C2, "val WER _r"), ("val_wer_n", C3, "val WER _n")]:
            ax.plot(e, [100 * r[key] for r in epochs], color=c, linewidth=2, marker="o", markersize=6, label=lab)
        ax.set(title="WER trên val theo epoch (epoch 0 = model gốc)", xlabel="epoch", ylabel="WER %")
        ax.legend(); fig.tight_layout()
        fig.savefig(pdir / "val_wer.png", dpi=130); plt.show() if show else None; plt.close(fig)

def plot_compare(run, table, fname, title):
    names = list(table); x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(9, 4))
    for j, (key, c, lab) in enumerate([("base", C1, "model gốc"), ("ft", C2, "fine-tuned")]):
        vals = [100 * table[n][key] for n in names]
        ax.bar(x + (j - 0.5) * 0.38, vals, width=0.36, color=c, label=lab)
        for xi, v in zip(x, vals):
            ax.text(xi + (j - 0.5) * 0.38, v + 0.8, f"{v:.0f}", ha="center", fontsize=8, color=INK2)
    ax.set_xticks(x, names, fontsize=8); ax.set(title=title, ylabel="WER %")
    ax.grid(axis="x", visible=False); ax.legend(); fig.tight_layout()
    fig.savefig(run / "plots" / fname, dpi=130); plt.show(); plt.close(fig)

def write_csv(path, rows):
    if rows:
        with open(path, "w", newline="", encoding="utf8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

## 8. Tạo thư mục run (đặt tên theo timestamp) và log

In [20]:
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN = RUNS_DIR / f"{STAMP}_ASR-{CFG['run_name']}"
for sub in ("plots", "checkpoints", "eval", "aug_samples"):
    (RUN / sub).mkdir(parents=True, exist_ok=True)

log = logging.getLogger("asr_ft"); log.setLevel(logging.INFO); log.handlers.clear()
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%H:%M:%S")
for h in (logging.FileHandler(RUN / "train.log", encoding="utf8"), logging.StreamHandler()):
    h.setFormatter(fmt); log.addHandler(h)
log.propagate = False

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
log.info(f"run dir: {RUN}")
log.info(f"gpu: {GPU} | amp: {'bf16' if BF16 else 'fp16'}")

20:07:58 | INFO | run dir: /content/drive/MyDrive/hospital_asr/runs/20261110_200758_ASR-PhoASR-small-hospitalnoise-synth
20:07:58 | INFO | gpu: Tesla T4 | amp: bf16


## 9. Nạp audio và nghe thử mẫu đã augment

In [21]:
modes = set(CFG["modes"].split(","))
train = [r for r in read_split("train") if r["mode"] in modes or r["mode"] == "u"]
val, test, test_new = read_split("val"), read_split("test"), read_split("test_newtext")
audio = {r["audio_filepath"]: load_wav(DATA / r["audio_filepath"]) for r in train + val + test + test_new}
log.info(f"train {len(train)} utts ({sum(r['duration'] for r in train) / 60:.1f} min) | val {len(val)} | "
         f"test {len(test)} | test_newtext {len(test_new)}")

noise_files = [load_wav(f) for f in sorted((DATA / "noise").glob("*.wav"))]
noise = HospitalNoise([audio[r["audio_filepath"]] for r in train], noise_files)
augment = Augmenter(noise, CFG)
log.info(f"noise kinds: {noise.kinds}")

rng = np.random.default_rng(CFG["seed"])
for i, r in enumerate(rng.choice(train, size=6, replace=False)):
    y = augment(audio[r["audio_filepath"]], np.random.default_rng((CFG["seed"], 999, i)))
    sf.write(RUN / "aug_samples" / f"{i}_{r['utt_id']}.wav", y, SR)
    if i < 3:
        print(r["utt_id"], "|", r["text"])
        display(Audio(audio[r["audio_filepath"]], rate=SR), Audio(y, rate=SR))

⚠️ train.jsonl: bỏ 1 dòng trùng audio_filepath


20:08:15 | INFO | train 9296 utts (546.8 min) | val 667 | test 1726 | test_newtext 267
20:08:15 | INFO | noise kinds: ['babble', 'hvac', 'beeps', 'trolley', 'file']


syn_t11950770_0 | Robot ơi, cho con hỏi nhà thuốc có bán khẩu trang không?


bud500_2_01167 | không được tơ hào gì tới nó hết cho đến


syn_t50169087_0 | Cho hỏi phòng khám mắt ở đâu thế?


## 10. Nạp model

In [22]:
processor = WhisperProcessor.from_pretrained(CFG["model"])
processor.tokenizer.set_prefix_tokens(language="vietnamese", task="transcribe", predict_timestamps=False)
model = WhisperForConditionalGeneration.from_pretrained(CFG["model"])
model.config.apply_spec_augment = CFG["spec_augment"] > 0
model.config.mask_time_prob = CFG["spec_augment"]
model.config.mask_feature_prob = CFG["spec_augment"]
model.config.forced_decoder_ids = None
model.generation_config.forced_decoder_ids = None
if CFG["freeze_encoder"]:
    model.freeze_encoder()
if CFG["grad_checkpointing"]:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
model.to(DEVICE)
log.info(f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")

collate = Collator(processor, CFG["n_weight"])
train_ds = SpeechDataset(train, audio, augment, CFG["seed"])
train_dl = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, collate_fn=collate,
                      num_workers=CFG["num_workers"], generator=torch.Generator().manual_seed(CFG["seed"]))
val_dl = DataLoader(SpeechDataset(val, audio, None, 0), batch_size=CFG["eval_batch_size"], collate_fn=collate)

steps_per_epoch = math.ceil(len(train_dl) / CFG["grad_accum"])
total_steps = steps_per_epoch * CFG["epochs"]
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup_ratio"] * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=not BF16)

config = dict(CFG, gpu=GPU, amp="bf16" if BF16 else "fp16", run_dir=str(RUN), train_utts=len(train),
              val_utts=len(val), test_utts=len(test), steps_per_epoch=steps_per_epoch, total_steps=total_steps,
              noise_kinds=noise.kinds, torch=torch.__version__, started=datetime.now().isoformat(timespec="seconds"))
(RUN / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=1), encoding="utf8")
log.info(f"steps/epoch {steps_per_epoch}, total steps {total_steps}, effective batch {CFG['batch_size'] * CFG['grad_accum']}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.86k [00:00<?, ?B/s]

20:08:40 | INFO | trainable params: 241.7M
20:08:40 | INFO | steps/epoch 581, total steps 3486, effective batch 16


## 11. Train

Epoch 0 là model gốc trên val. Checkpoint tốt nhất được chọn theo **WER `_r` trên val**, vì nhãn `_r` là chính xác. Loss curve được ghi lại sau mỗi epoch.

In [ ]:
val_waves = [audio[r["audio_filepath"]] for r in val]

def evaluate_val(epoch, step, train_loss):
    t0 = time.time()
    vl = val_loss(model, val_dl)
    m = score(val, transcribe(model, processor, val_waves, CFG["eval_batch_size"]))
    row = dict(epoch=epoch, step=step, train_loss=round(train_loss, 4), val_loss=round(vl, 4),
               **{f"val_{k}": v for k, v in m.items()}, eval_sec=round(time.time() - t0, 1))
    log.info(f"[epoch {epoch}] train_loss {train_loss:.4f} | val_loss {vl:.4f} | val WER {m['wer_all']:.2%} "
             f"(_r {m.get('wer_r', float('nan')):.2%}, _n {m.get('wer_n', float('nan')):.2%}) | CER {m['cer_all']:.2%}")
    return row

def save_ckpt(name):
    model.save_pretrained(RUN / "checkpoints" / name)
    processor.save_pretrained(RUN / "checkpoints" / name)

step_rows, epoch_rows = [], [evaluate_val(0, 0, float("nan"))]
best_wer, best_epoch, bad = epoch_rows[0]["val_wer_r"], 0, 0
save_ckpt("best")

step, t_last = 0, time.time()
for epoch in range(1, CFG["epochs"] + 1):
    train_ds.epoch = epoch
    model.train()
    run_loss, run_n = 0.0, 0
    opt.zero_grad(set_to_none=True)
    for i, b in enumerate(train_dl):
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            out = model(input_features=b["input_features"].to(DEVICE), labels=b["labels"].to(DEVICE))
        loss = weighted_loss(out.logits, b["labels"].to(DEVICE), b["weights"].to(DEVICE))
        scaler.scale(loss / CFG["grad_accum"]).backward()
        run_loss += loss.item(); run_n += 1
        if (i + 1) % CFG["grad_accum"] and i + 1 != len(train_dl):
            continue
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"]).item()
        scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
        step += 1
        now = time.time()
        step_rows.append(dict(step=step, epoch=round(epoch - 1 + (i + 1) / len(train_dl), 3), lr=sched.get_last_lr()[0],
                              train_loss=round(loss.item(), 4), grad_norm=round(gnorm, 3), sec_per_step=round(now - t_last, 2)))
        t_last = now
        if step % CFG["log_every"] == 0:
            log.info(f"step {step}/{total_steps} ep {epoch} | loss {loss.item():.4f} | lr {sched.get_last_lr()[0]:.2e} "
                     f"| gnorm {gnorm:.2f} | {step_rows[-1]['sec_per_step']}s/step")

    row = evaluate_val(epoch, step, run_loss / max(run_n, 1))
    epoch_rows.append(row)
    write_csv(RUN / "metrics_steps.csv", step_rows)
    write_csv(RUN / "metrics_epochs.csv", epoch_rows)
    plot_curves(RUN, step_rows, epoch_rows)
    if row["val_wer_r"] < best_wer - 1e-4:
        best_wer, best_epoch, bad = row["val_wer_r"], epoch, 0
        save_ckpt("best")
        log.info(f"  new best val WER_r {best_wer:.2%} -> checkpoints/best")
    else:
        bad += 1
        log.info(f"  no improvement ({bad}/{CFG['patience']}), best {best_wer:.2%} @ epoch {best_epoch}")
        if bad >= CFG["patience"]:
            log.info("early stop")
            break

save_ckpt("last")
log.info(f"training done. best epoch {best_epoch}, val WER_r {best_wer:.2%}")

## 12. Đường loss và WER

In [ ]:
plot_curves(RUN, step_rows, epoch_rows, show=True)
pd.DataFrame(epoch_rows)

## 13. Đánh giá cuối: model gốc và checkpoint tốt nhất

Chạy trên **test** (v1: 6 người nói chưa thấy; v2: toàn bộ 1726 câu thật của 29 người, vì không câu thật nào được dùng để train) ở ba điều kiện: sạch, thêm ồn bệnh viện 10 dB và 5 dB. Ồn được sinh cố định theo từng câu, nên hai model nhận cùng một đầu vào. Ngoài ra chạy trên **test_newtext** (câu chưa thấy) để kiểm tra model có học thuộc câu không. Với v2 có thêm `test_by_old_split`: dòng `old_split=test` là 6 người nói mà model v1 cũng chưa thấy, dùng để so công bằng với model v1 (WER 9,4%).

In [ ]:
snrs = [float(s) for s in CFG["test_snrs"].split(",") if s]
eval_noise = HospitalNoise([audio[r["audio_filepath"]] for r in train], noise_files)
conditions = {"sạch": [audio[r["audio_filepath"]] for r in test]}
for snr in snrs:
    conditions[f"ồn {snr:g} dB"] = [
        add_noise(audio[r["audio_filepath"]], eval_noise(len(audio[r["audio_filepath"]]), np.random.default_rng((7, int(snr), i)), n_layers=2), snr)
        for i, r in enumerate(test)]

del model, opt; torch.cuda.empty_cache()
models = {"base": CFG["model"], "ft": str(RUN / "checkpoints" / "best")}
final = dict(best_epoch=best_epoch, best_val_wer_r=best_wer, test_by_condition={}, test_by_speaker={}, test_newtext={}, summary={})
preds = {}
for tag, path in models.items():
    m = WhisperForConditionalGeneration.from_pretrained(path).to(DEVICE)
    for cname, waves in conditions.items():
        hyps = transcribe(m, processor, waves, CFG["eval_batch_size"])
        preds[(tag, cname)] = hyps
        s = score(test, hyps)
        final["test_by_condition"].setdefault(cname, {})[tag] = s["wer_all"]
        final["summary"][f"{tag} | test {cname}"] = s
        log.info(f"  {tag:4s} test {cname:9s} WER {s['wer_all']:.2%} (_r {s['wer_r']:.2%}) CER {s['cer_all']:.2%}")
    hyps = transcribe(m, processor, [audio[r["audio_filepath"]] for r in test_new], CFG["eval_batch_size"])
    preds[(tag, "newtext")] = hyps
    s = score(test_new, hyps)
    final["test_newtext"][tag] = s
    final["summary"][f"{tag} | test_newtext"] = s
    log.info(f"  {tag:4s} test_newtext    WER {s['wer_all']:.2%} (_r {s.get('wer_r', float('nan')):.2%})")
    del m; torch.cuda.empty_cache()

for spk in sorted({r["speaker_id"] for r in test}):
    ix = [i for i, r in enumerate(test) if r["speaker_id"] == spk]
    final["test_by_speaker"][spk] = {tag: round(jiwer.wer([normalize(test[i]["text"]) for i in ix],
                                                          [normalize(preds[(tag, "sạch")][i]) for i in ix]), 4) for tag in models}

# v2: test = mọi người nói thật. old_split cho biết người đó thuộc split nào ở bản v1:
#   old_split == "test" -> 6 người nói mà model v1 cũng chưa từng thấy (so sánh công bằng với model v1)
final["test_by_old_split"] = {}
for osplit in sorted({r.get("old_split", "") for r in test} - {"", "nan"}):
    ix = [i for i, r in enumerate(test) if r.get("old_split") == osplit]
    final["test_by_old_split"][osplit] = {
        tag: {c: score([test[i] for i in ix], [preds[(tag, c)][i] for i in ix])["wer_all"] for c in conditions}
        for tag in models}
    log.info(f"  old_split={osplit:5s} ({len(ix)} utts): " + " | ".join(
        f"{tag} " + ", ".join(f"{c} {v:.2%}" for c, v in final['test_by_old_split'][osplit][tag].items()) for tag in models))

rows = []
for cname in conditions:
    for i, r in enumerate(test):
        ref = normalize(r["text"]); b, ft = preds[("base", cname)][i], preds[("ft", cname)][i]
        rows.append(dict(utt_id=r["utt_id"], speaker_id=r["speaker_id"], mode=r["mode"], condition=cname, reference=r["text"],
                         base_hyp=b, ft_hyp=ft, base_wer=round(jiwer.wer(ref, normalize(b)), 4), ft_wer=round(jiwer.wer(ref, normalize(ft)), 4)))
write_csv(RUN / "eval" / "test_predictions.csv", rows)
write_csv(RUN / "eval" / "test_newtext_predictions.csv", [
    dict(utt_id=r["utt_id"], speaker_id=r["speaker_id"], mode=r["mode"], reference=r["text"],
         base_hyp=preds[("base", "newtext")][i], ft_hyp=preds[("ft", "newtext")][i]) for i, r in enumerate(test_new)])
final["finished"] = datetime.now().isoformat(timespec="seconds")
(RUN / "eval" / "summary.json").write_text(json.dumps(final, ensure_ascii=False, indent=1), encoding="utf8")

plot_compare(RUN, final["test_by_condition"], "test_by_snr.png", "WER trên test: sạch và có thêm ồn bệnh viện")
plot_compare(RUN, final["test_by_speaker"], "test_by_speaker.png", "WER trên test theo người nói")
pd.DataFrame({k: {"WER": f"{s['wer_all']:.2%}", "WER _r": f"{s.get('wer_r', float('nan')):.2%}",
                  "CER": f"{s['cer_all']:.2%}", "câu đúng": f"{s['sent_acc']:.2%}"}
              for k, s in final["summary"].items()}).T

## 14. Xem các câu được sửa và các câu bị hỏng

In [ ]:
df = pd.DataFrame(rows)
clean = df[df.condition == "sạch"]
print("Câu được sửa (base sai, fine-tuned đúng):")
display(clean[(clean.base_wer > 0) & (clean.ft_wer == 0)][["utt_id", "reference", "base_hyp", "ft_hyp"]].head(15))
print("Câu bị hỏng (base đúng, fine-tuned sai):")
display(clean[(clean.base_wer == 0) & (clean.ft_wer > 0)][["utt_id", "reference", "base_hyp", "ft_hyp"]].head(15))
print(f"\nMọi file của run này: {RUN}")

## 15. (Tuỳ chọn) Tải checkpoint về máy

Checkpoint đã nằm trong Drive tại `runs/<run>/checkpoints/best`. Muốn dùng ở local thì chép cả thư mục đó vào `models_finetuned/` của repo, rồi đánh giá lại:

```bash
sim_vision/python/.venv/bin/python scripts/eval_asr_testset.py --model models_finetuned/<run>/best --run-name <run>_test
```

In [ ]:
# shutil.make_archive(f"/content/{RUN.name}_best", "zip", RUN / "checkpoints" / "best")
# from google.colab import files; files.download(f"/content/{RUN.name}_best.zip")